## Interactive Entity Mapping for PII Evaluation

This notebook explains how to use the `EntityMappingHelper` to align entity types between your dataset and the model being evaluated. Proper entity mapping is crucial for accurate evaluation metrics.

**Topics covered:**
1. Why entity mapping is needed
2. Using the EntityMappingHelper
3. Understanding the mapping review output
4. What mapping to `None` means (and its impact on evaluation)
5. Advanced mapping customization with custom mappers
6. How `entities_to_keep` influences the experiment

**Prerequisites:** Basic familiarity with Presidio Evaluator concepts from notebooks 4 and 5.

In [1]:
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import List, Dict

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator
from presidio_evaluator.models import PresidioAnalyzerWrapper
from presidio_evaluator.entity_mapping import (
    EntityMappingHelper,
    SemanticEntityMapper,
    create_presidio_mapper,
    suggest_mapping,
)

from presidio_analyzer import AnalyzerEngine

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

%reload_ext autoreload
%autoreload 2

## 1. Why Entity Mapping is Needed

When evaluating a PII detection model, you're comparing:
- **Dataset entities**: The ground truth labels in your evaluation dataset (e.g., `STREET_ADDRESS`, `FIRST_NAME`, `GPE`)
- **Model entities**: The entity types the model can detect (e.g., `LOCATION`, `PERSON`, `NRP`)

These often use **different naming conventions**:

| Dataset Entity | Model Entity | Notes |
|---------------|--------------|-------|
| `STREET_ADDRESS` | `LOCATION` | Different granularity |
| `GPE` (geopolitical entity) | `LOCATION` | Geographic entities |
| `FIRST_NAME`, `LAST_NAME` | `PERSON` | Model groups all names |
| `IP_ADDRESS` | `IP_ADDRESS` | Exact match ✓ |
| `INTERNAL_CODE` | ❌ | Model doesn't support this |

Without proper mapping, the evaluator would count correct predictions as errors because the entity names don't match.

In [2]:
# Load dataset
dataset_name = "synth_dataset_v2.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))
print(f"Loaded {len(dataset)} samples")

# Show dataset entities
dataset_entities = Counter()
for sample in dataset:
    for span in sample.spans:
        dataset_entities[span.entity_type] += 1

print(f"\nDataset has {len(dataset_entities)} entity types:")
pprint(dict(dataset_entities.most_common(10)), compact=True)

tokenizing input:   0%|          | 0/1500 [00:00<?, ?it/s]

loading model en_core_web_sm


tokenizing input: 100%|██████████| 1500/1500 [00:04<00:00, 314.37it/s]

Loaded 1500 samples

Dataset has 17 entity types:
{'AGE': 74,
 'CREDIT_CARD': 136,
 'DATE_TIME': 119,
 'GPE': 411,
 'NRP': 55,
 'ORGANIZATION': 250,
 'PERSON': 857,
 'PHONE_NUMBER': 92,
 'STREET_ADDRESS': 598,
 'TITLE': 92}


In [3]:
# Load Presidio Analyzer
analyzer = AnalyzerEngine(default_score_threshold=0.4)

print(f"Model supports {len(analyzer.get_supported_entities('en'))} entity types:")
pprint(sorted(analyzer.get_supported_entities('en')), compact=True)

Model supports 19 entity types:
['CREDIT_CARD', 'CRYPTO', 'DATE_TIME', 'EMAIL_ADDRESS', 'IBAN_CODE',
 'IP_ADDRESS', 'LOCATION', 'MAC_ADDRESS', 'MEDICAL_LICENSE', 'NRP', 'PERSON',
 'PHONE_NUMBER', 'UK_NHS', 'URL', 'US_BANK_NUMBER', 'US_DRIVER_LICENSE',
 'US_ITIN', 'US_PASSPORT', 'US_SSN']


## 2. Using the EntityMappingHelper

The `EntityMappingHelper` automates the mapping process:

1. **Auto-detects** entities from both dataset and model
2. **Suggests mappings** using semantic similarity (e.g., `FIRST_NAME` → `PERSON`)
3. **Highlights unmapped** entities that need your attention
4. **Allows manual adjustments** to fine-tune the mapping

In [4]:
# Create the helper - automatically detects entities and suggests mappings
helper = EntityMappingHelper(
    dataset=dataset,
    model=analyzer,
    language="en",
    threshold=0.4  # Minimum similarity score for auto-mapping (0-1)
)

# Review the suggested mapping
helper.review_mapping()

Dataset Entity,→ Model Entity,Samples,Confidence
⚠️ AGE,NOT MAPPED,74,0.00
⚠️ ORGANIZATION,NOT MAPPED,199,0.00
⚠️ TITLE,NOT MAPPED,92,0.00
✓ CREDIT_CARD,CREDIT_CARD,136,1.00
✓ DATE_TIME,DATE_TIME,119,1.00
✓ DOMAIN_NAME,URL,37,1.00
✓ EMAIL_ADDRESS,EMAIL_ADDRESS,49,1.00
✓ GPE,LOCATION,325,1.00
✓ IBAN_CODE,IBAN_CODE,21,1.00
✓ IP_ADDRESS,IP_ADDRESS,14,1.00


## 3. Understanding the Mapping Review Output

The `review_mapping()` display shows:

### Status Indicators
| Icon | Meaning |
|------|--------|
| ✓ (green row) | High confidence mapping (≥0.70) |
| ⚠ (yellow row) | Low confidence mapping (needs review) |
| ⚠️ (red "NOT MAPPED") | Entity has no mapping - **action required** |
| ○ (gray row) | Explicitly mapped to `None` |

### Confidence Scores
- **1.00 ✎ (blue)**: Manual mapping you set
- **None ✎ (gray)**: Explicitly mapped to `None`
- **≥0.70 (green)**: High confidence auto-mapping
- **0.01-0.69 (yellow)**: Low confidence - consider reviewing
- **0.00 (red)**: Unmapped - requires action

### Collapsible Sections
- **📊 Dataset Entities**: All entities in your dataset with sample counts
- **📦 Model Entities**: All entities the model can detect

In [5]:
# Alternative: compact text-based review (useful for scripts/logging)
helper.review_mapping(format="compact")


MAPPING: 14 mapped, 3 unmapped | Dataset: 17 active, 0 excluded | Model: 19 entities

⚠️  UNMAPPED (3):
  • AGE (74 samples)
  • ORGANIZATION (199 samples)
  • TITLE (92 samples)
  💡 You can: 1) map to a model entity, 2) map to None (FN penalty), or 3) exclude them.


## 4. What Does Mapping to `None` Mean?

When you map a dataset entity to `None`, you're saying:

> "This entity exists in my dataset, but my model **cannot detect it**. 
> I want to **keep it in evaluation** and **penalize the model** for missing it."

### Mapping to `None` vs Excluding

| Action | Effect on Evaluation |
|--------|---------------------|
| `set_mapping("ENTITY", None)` | Entity stays in dataset. Model **cannot detect it**, so all instances become **False Negatives** (FN). This lowers recall. |
| `exclude_dataset_entities(["ENTITY"])` | Entity is **removed** from evaluation entirely. Not counted in any metrics. |

### When to Use Each

**Map to `None`** when:
- The entity represents real PII that your model *should* detect but doesn't
- You want to capture the "true" recall of your model
- Example: Model doesn't support `AGE`, but age is PII you care about

**Exclude** when:
- The entity is irrelevant to your use case
- The entity is noise or shouldn't be considered PII
- Example: `DEBUG_INFO` or `INTERNAL_CODE` that isn't real PII

In [6]:
# Example: Map TITLE to None (model doesn't detect titles like "Mr.", "Dr.")
# This means all TITLE instances will be False Negatives

helper.set_mapping("TITLE", None)

# Review to see the change
helper.review_mapping(format="compact")

✓ Mapping set: TITLE → None
   (2 entities still unmapped)

MAPPING: 14 mapped, 2 unmapped | Dataset: 17 active, 0 excluded | Model: 19 entities

⚠️  UNMAPPED (2):
  • AGE (74 samples)
  • ORGANIZATION (199 samples)
  💡 You can: 1) map to a model entity, 2) map to None (FN penalty), or 3) exclude them.

🔧 Manual (1): TITLE→None


In [7]:
# Set additional mappings to resolve unmapped entities

# Map AGE to DATE_TIME (Presidio detects some ages as dates)
helper.set_mapping("AGE", "DATE_TIME")

# If ORGANIZATION is unmapped, we might map it to None
# (Presidio doesn't have a dedicated ORGANIZATION recognizer by default)
helper.set_mapping("ORGANIZATION", None)

helper.review_mapping()

✓ Mapping set: AGE → DATE_TIME
   (1 entities still unmapped)
✓ Mapping set: ORGANIZATION → None


Dataset Entity,→ Model Entity,Samples,Confidence
○ ORGANIZATION,None,199,None ✎
○ TITLE,None,92,None ✎
✓ CREDIT_CARD,CREDIT_CARD,136,1.00
✓ DATE_TIME,DATE_TIME,119,1.00
✓ DOMAIN_NAME,URL,37,1.00
✓ EMAIL_ADDRESS,EMAIL_ADDRESS,49,1.00
✓ GPE,LOCATION,325,1.00
✓ IBAN_CODE,IBAN_CODE,21,1.00
✓ IP_ADDRESS,IP_ADDRESS,14,1.00
✓ PERSON,PERSON,637,1.00


### Impact of Mapping to `None` on Metrics

Let's demonstrate how mapping to `None` affects your evaluation metrics.

In [8]:
# Count how many samples will be affected by None mappings
none_mapped_entities = [
    entity for entity, target in helper._suggested_mapping.items() 
    if target is None and entity in helper._manual_mappings
]

print("Entities mapped to None (will become False Negatives):")
for entity in none_mapped_entities:
    count = helper._dataset_entity_counts.get(entity, 0)
    print(f"  • {entity}: {count} samples → all will be FN")

total_fn_from_none = sum(
    helper._dataset_entity_counts.get(e, 0) 
    for e in none_mapped_entities
)
print(f"\nTotal False Negatives from None mappings: {total_fn_from_none}")

Entities mapped to None (will become False Negatives):
  • ORGANIZATION: 199 samples → all will be FN
  • TITLE: 92 samples → all will be FN

Total False Negatives from None mappings: 291


## 5. Advanced: Customizing the Mapping Logic

The `EntityMappingHelper` uses a **semantic similarity mapper** by default. You can customize this behavior in several ways:

### Default Mapper: Presidio Mapper
- Combines exact matching with semantic similarity
- Uses predefined mappings for common Presidio entity patterns
- Falls back to semantic similarity for unknown entities

### Custom Options:
1. **Adjust the threshold** - higher = stricter matching
2. **Use SemanticEntityMapper directly** - for pure semantic matching
3. **Create your own mapper** - full control over mapping logic

In [9]:
# Example: Using SemanticEntityMapper directly
# This uses sentence embeddings to find similar entity names

semantic_mapper = SemanticEntityMapper(
    threshold=0.35,  # Minimum similarity score for a match
    full_name_weight=0.6  # Weight for full name vs word-level similarity
)

# Test individual mappings
print("Semantic similarity examples:")
test_cases = [
    ("FIRST_NAME", ["PERSON", "LOCATION", "DATE_TIME"]),
    ("STREET_ADDRESS", ["LOCATION", "PERSON", "EMAIL_ADDRESS"]),
    ("SSN", ["US_SSN", "PERSON", "LOCATION"]),
    ("BIRTHDAY", ["DATE_TIME", "PERSON", "LOCATION"]),
]

for source, targets in test_cases:
    result = semantic_mapper.map(source, targets)
    print(f"  {source} → {result}")

Semantic similarity examples:
  FIRST_NAME → DATE_TIME
  STREET_ADDRESS → EMAIL_ADDRESS
  SSN → US_SSN
  BIRTHDAY → PERSON


In [10]:
# Pass the custom SemanticEntityMapper into EntityMappingHelper
# This replaces the default Presidio mapper entirely

semantic_mapper = SemanticEntityMapper(
    threshold=0.35,
    full_name_weight=0.6
)

semantic_helper = EntityMappingHelper(
    dataset=dataset,
    model=analyzer,
    language="en",
    mapper=semantic_mapper,  # Custom mapper instead of create_presidio_mapper()
)

print("Mapping using SemanticEntityMapper:")
semantic_helper.review_mapping(format="compact")


Mapping using SemanticEntityMapper:

MAPPING: 13 mapped, 4 unmapped | Dataset: 17 active, 0 excluded | Model: 19 entities

⚠️  UNMAPPED (4):
  • AGE (74 samples)
  • GPE (325 samples)
  • ORGANIZATION (199 samples)
  • TITLE (92 samples)
  💡 You can: 1) map to a model entity, 2) map to None (FN penalty), or 3) exclude them.


In [11]:
# Create a helper with a stricter threshold
strict_helper = EntityMappingHelper(
    dataset=dataset,
    model=analyzer,
    language="en",
    threshold=0.7  # Higher threshold = fewer auto-mappings
)

print("With strict threshold (0.7), fewer entities are auto-mapped:")
strict_helper.review_mapping(format="compact")

With strict threshold (0.7), fewer entities are auto-mapped:

MAPPING: 14 mapped, 3 unmapped | Dataset: 17 active, 0 excluded | Model: 19 entities

⚠️  UNMAPPED (3):
  • AGE (74 samples)
  • ORGANIZATION (199 samples)
  • TITLE (92 samples)
  💡 You can: 1) map to a model entity, 2) map to None (FN penalty), or 3) exclude them.


In [12]:
# You can also use suggest_mapping() directly for programmatic access

dataset_entities = list(set(
    span.entity_type for sample in dataset for span in sample.spans
))
model_entities = analyzer.get_supported_entities("en")

mapping, scores = suggest_mapping(
    dataset_entities=dataset_entities,
    model_entities=model_entities,
    return_scores=True
)

print("Programmatic mapping with confidence scores:")
for entity in sorted(dataset_entities):
    target = mapping.get(entity)
    score = scores.get(entity, 0)
    print(f"  {entity:20} → {str(target):20} (confidence: {score:.2f})")

Programmatic mapping with confidence scores:
  AGE                  → None                 (confidence: 0.00)
  CREDIT_CARD          → CREDIT_CARD          (confidence: 1.00)
  DATE_TIME            → DATE_TIME            (confidence: 1.00)
  DOMAIN_NAME          → URL                  (confidence: 1.00)
  EMAIL_ADDRESS        → EMAIL_ADDRESS        (confidence: 1.00)
  GPE                  → LOCATION             (confidence: 1.00)
  IBAN_CODE            → IBAN_CODE            (confidence: 1.00)
  IP_ADDRESS           → IP_ADDRESS           (confidence: 1.00)
  NRP                  → NRP                  (confidence: 1.00)
  ORGANIZATION         → None                 (confidence: 0.00)
  PERSON               → PERSON               (confidence: 1.00)
  PHONE_NUMBER         → PHONE_NUMBER         (confidence: 1.00)
  STREET_ADDRESS       → LOCATION             (confidence: 1.00)
  TITLE                → None                 (confidence: 0.00)
  US_DRIVER_LICENSE    → US_DRIVER_LICENSE   

## 6. How `entities_to_keep` Influences the Experiment

The `entities_to_keep` parameter controls **which model predictions are considered** during evaluation.

### Understanding `entities_to_keep`

When you set `entities_to_keep`:
- **Model predictions** for entities NOT in this list are **ignored** (treated as non-PII)
- **Dataset labels** are still evaluated based on the mapping

### Two Evaluation Strategies

| Strategy | `entities_to_keep` | Effect |
|----------|-------------------|--------|
| **Filtered** | `helper.get_model_entities_to_use()` | Only evaluate entities that exist in your dataset. Fair comparison with ground truth. |
| **Full Model** | `None` | Evaluate all model predictions. May increase recall but decrease precision (model might detect entities not in your dataset). |

### When to Use Each

**Filtered (Recommended for benchmarking):**
- When you want metrics that reflect the dataset's scope
- When comparing models on the same dataset
- When the dataset is comprehensive for your use case

**Full Model:**
- When you want to see the model's full capability
- When the dataset doesn't cover all entity types you care about
- When you want to discover what else the model can detect

In [13]:
# First, let's resolve any remaining unmapped entities
# (You need to do this before getting the final mapping)

# Check what's still unmapped
if helper._unmapped_entities:
    print("Resolving unmapped entities:")
    for entity in helper._unmapped_entities.copy():
        # For this example, we'll map unknown entities to None
        helper.set_mapping(entity, None)
else:
    print("All entities are already mapped!")

All entities are already mapped!


In [14]:
# Get the final mapping and entities
entity_mapping = helper.get_mapping()
model_entities_to_use = helper.get_model_entities_to_use()

print("Entity Mapping (dataset → model):")
pprint(entity_mapping, compact=True)

print(f"\nModel entities to use ({len(model_entities_to_use)}):")
pprint(sorted(model_entities_to_use), compact=True)

Entity Mapping (dataset → model):
{'AGE': 'DATE_TIME',
 'CREDIT_CARD': 'CREDIT_CARD',
 'DATE_TIME': 'DATE_TIME',
 'DOMAIN_NAME': 'URL',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'GPE': 'LOCATION',
 'IBAN_CODE': 'IBAN_CODE',
 'IP_ADDRESS': 'IP_ADDRESS',
 'NRP': 'NRP',
 'PERSON': 'PERSON',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'STREET_ADDRESS': 'LOCATION',
 'US_DRIVER_LICENSE': 'US_DRIVER_LICENSE',
 'US_SSN': 'US_SSN',
 'ZIP_CODE': 'LOCATION'}

Model entities to use (14):
['CREDIT_CARD', 'DATE_TIME', 'EMAIL_ADDRESS', 'IBAN_CODE', 'IP_ADDRESS',
 'LOCATION', 'NRP', 'ORGANIZATION', 'PERSON', 'PHONE_NUMBER', 'TITLE', 'URL',
 'US_DRIVER_LICENSE', 'US_SSN']


In [15]:
# Compare: What entities would be evaluated with each strategy?

all_model_entities = set(analyzer.get_supported_entities("en"))
filtered_entities = set(model_entities_to_use)

print("=== Evaluation Strategy Comparison ===")
print(f"\nFiltered strategy (entities_to_keep={len(filtered_entities)}):")
print(f"  Only these entities count: {sorted(filtered_entities)}")

excluded_from_eval = all_model_entities - filtered_entities
print(f"\n  Model entities IGNORED ({len(excluded_from_eval)}):")
print(f"  {sorted(excluded_from_eval)}")
print("  → Predictions of these types won't count as FP or TP")

print(f"\nFull model strategy (entities_to_keep=None):")
print(f"  All {len(all_model_entities)} model entities are evaluated")
print("  → Model might detect entities not in dataset → could increase FP")

=== Evaluation Strategy Comparison ===

Filtered strategy (entities_to_keep=14):
  Only these entities count: ['CREDIT_CARD', 'DATE_TIME', 'EMAIL_ADDRESS', 'IBAN_CODE', 'IP_ADDRESS', 'LOCATION', 'NRP', 'ORGANIZATION', 'PERSON', 'PHONE_NUMBER', 'TITLE', 'URL', 'US_DRIVER_LICENSE', 'US_SSN']

  Model entities IGNORED (7):
  ['CRYPTO', 'MAC_ADDRESS', 'MEDICAL_LICENSE', 'UK_NHS', 'US_BANK_NUMBER', 'US_ITIN', 'US_PASSPORT']
  → Predictions of these types won't count as FP or TP

Full model strategy (entities_to_keep=None):
  All 19 model entities are evaluated
  → Model might detect entities not in dataset → could increase FP


In [16]:
# Run evaluation with both strategies to see the difference

filtered_dataset = helper.get_filtered_dataset()
wrapped_analyzer = PresidioAnalyzerWrapper(
    analyzer_engine=analyzer,
    language="en"
)

# Strategy 1: Filtered entities
evaluator_filtered = SpanEvaluator(
    model=wrapped_analyzer,
    entity_mapping=entity_mapping,
    entities_to_keep=model_entities_to_use,  # Only mapped entities
)

# Strategy 2: All model entities
evaluator_full = SpanEvaluator(
    model=wrapped_analyzer,
    entity_mapping=entity_mapping,
    entities_to_keep=None,  # All entities
)

print("Running evaluation with both strategies...")
print("(This may take a moment)")

--------
Entities supported by this Presidio Analyzer instance:
US_SSN, US_ITIN, EMAIL_ADDRESS, DATE_TIME, LOCATION, CRYPTO, US_PASSPORT, URL, PERSON, CREDIT_CARD, US_DRIVER_LICENSE, MEDICAL_LICENSE, UK_NHS, NRP, MAC_ADDRESS, PHONE_NUMBER, IP_ADDRESS, IBAN_CODE, US_BANK_NUMBER
Running evaluation with both strategies...
(This may take a moment)


/Users/omrimendels/Documents/github/presidio-research/presidio_evaluator/evaluation/base_evaluator.py:103: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn(


In [17]:
%%time

# Use a subset for faster comparison
subset = filtered_dataset[:100]

# Evaluate with filtered entities
results_filtered = evaluator_filtered.evaluate_all(subset)
scores_filtered = evaluator_filtered.calculate_score(results_filtered)

# Evaluate with all entities
results_full = evaluator_full.evaluate_all(subset)
scores_full = evaluator_full.calculate_score(results_full)

# Compare results
print("\n=== Strategy Comparison (on 100 samples) ===")
print(f"\n{'Metric':<20} {'Filtered':>12} {'Full Model':>12}")
print("-" * 44)
print(f"{'PII Precision':<20} {scores_filtered.pii_precision:>12.3f} {scores_full.pii_precision:>12.3f}")
print(f"{'PII Recall':<20} {scores_filtered.pii_recall:>12.3f} {scores_full.pii_recall:>12.3f}")
print(f"{'PII F1':<20} {scores_filtered.pii_f:>12.3f} {scores_full.pii_f:>12.3f}")


=== Strategy Comparison (on 100 samples) ===

Metric                   Filtered   Full Model
--------------------------------------------
PII Precision               0.670        0.727
PII Recall                  0.584        0.640
PII F1                      0.600        0.656
CPU times: user 980 ms, sys: 34.2 ms, total: 1.01 s
Wall time: 1.02 s


### Excluding Specific Model Entities

You can also exclude specific model entities to prevent false positives from entity types that aren't relevant to your evaluation.

In [18]:
# Example: Exclude model entities not in our dataset
# This prevents FPs from entities the dataset doesn't have

# Create a fresh helper
helper2 = EntityMappingHelper(
    dataset=dataset,
    model=analyzer,
    language="en"
)

# Get model entities that are NOT mapped from the dataset
mapped_model_entities = set(helper2._suggested_mapping.values()) - {None}
all_model_entities = set(analyzer.get_supported_entities("en"))
unused_model_entities = all_model_entities - mapped_model_entities

print(f"Model entities not used by dataset ({len(unused_model_entities)}):")
pprint(sorted(unused_model_entities), compact=True)

# Exclude these to prevent spurious FPs
if unused_model_entities:
    helper2.exclude_model_entities(list(unused_model_entities))

Model entities not used by dataset (7):
['CRYPTO', 'MAC_ADDRESS', 'MEDICAL_LICENSE', 'UK_NHS', 'US_BANK_NUMBER',
 'US_ITIN', 'US_PASSPORT']
✓ Excluded 7 model entity(ies)


## Summary

### Key Takeaways

1. **Entity mapping is essential** for accurate evaluation when dataset and model use different entity names.

2. **`EntityMappingHelper`** automates the mapping process:
   - Auto-detects entities from dataset and model
   - Suggests mappings using semantic similarity
   - Allows manual adjustments with `set_mapping()`

3. **Mapping to `None`** means:
   - The model cannot detect this entity type
   - All instances will count as **False Negatives**
   - Use this when the entity is real PII you care about

4. **Excluding entities** removes them from evaluation entirely:
   - Use `exclude_dataset_entities()` for irrelevant dataset entities
   - Use `exclude_model_entities()` to ignore certain model predictions

5. **`entities_to_keep`** controls which model predictions are evaluated:
   - Filtered mode: Fair comparison with dataset (recommended)
   - Full model mode: See all model capabilities

### Best Practices

1. Always review mappings with `helper.review_mapping()` before running experiments
2. Explicitly handle unmapped entities (map to a target, map to `None`, or exclude)
3. Document your mapping decisions for reproducibility
4. Use the experiment tracker to log your entity mappings

In [19]:
# Quick Reference: EntityMappingHelper API

print("""
=== EntityMappingHelper Quick Reference ===

# Create helper
helper = EntityMappingHelper(dataset=dataset, model=model, language="en")

# Review current mapping
helper.review_mapping()                    # HTML display (interactive)
helper.review_mapping(format="compact")   # Text display (scripts)

# Manual adjustments
helper.set_mapping("DATASET_ENTITY", "MODEL_ENTITY")  # Map to model entity
helper.set_mapping("DATASET_ENTITY", None)             # Model can't detect (→FN)

# Exclusions
helper.exclude_dataset_entities(["ENTITY1", "ENTITY2"])  # Remove from dataset
helper.exclude_model_entities(["ENTITY1", "ENTITY2"])    # Ignore predictions

# Get final configuration
mapping = helper.get_mapping()                  # Dict: dataset → model entities
entities = helper.get_model_entities_to_use()  # List: entities to evaluate
dataset = helper.get_filtered_dataset()        # Filtered dataset samples
""")


=== EntityMappingHelper Quick Reference ===

# Create helper
helper = EntityMappingHelper(dataset=dataset, model=model, language="en")

# Review current mapping
helper.review_mapping()                    # HTML display (interactive)
helper.review_mapping(format="compact")   # Text display (scripts)

# Manual adjustments
helper.set_mapping("DATASET_ENTITY", "MODEL_ENTITY")  # Map to model entity
helper.set_mapping("DATASET_ENTITY", None)             # Model can't detect (→FN)

# Exclusions
helper.exclude_dataset_entities(["ENTITY1", "ENTITY2"])  # Remove from dataset
helper.exclude_model_entities(["ENTITY1", "ENTITY2"])    # Ignore predictions

# Get final configuration
mapping = helper.get_mapping()                  # Dict: dataset → model entities
entities = helper.get_model_entities_to_use()  # List: entities to evaluate
dataset = helper.get_filtered_dataset()        # Filtered dataset samples

